# Task 1: Video Pre-processing & Keyframe Extraction (SBD)

**Mục tiêu:** So sánh phương pháp trích xuất khung hình hiện tại (Uniform/OpenCV) với SOTA (TransNetV2 / AutoShot).

Notebook này được thiết kế để chạy trên Kaggle GPU T4.

In [1]:
# 1. Cài đặt các thư viện cần thiết
!pip install opencv-python pillow ffmpeg-python

# Tải mô hình TransNetV2 từ GitHub để test
!git clone https://github.com/soCzech/TransNetV2.git
import sys
sys.path.append('/kaggle/working/TransNetV2/inference')

Cloning into 'TransNetV2'...
remote: Enumerating objects: 362, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 362 (delta 71), reused 71 (delta 71), pack-reused 274 (from 1)
Receiving objects: 100% (362/362), 95.25 KiB | 5.29 MiB/s, done.
Resolving deltas: 100% (210/210), done.
Filtering content: 100% (3/3), 34.77 MiB | 8.27 MiB/s, done.


In [2]:
import cv2
import time
import os

# Định nghĩa đường dẫn video đầu vào (Thay đổi bằng file thật trên Kaggle của bạn)
video_path = '/kaggle/input/datasets/phmthanhhng27/video-l30/video/L30_V003.mp4'

# --- P1: Baseline (OpenCV Uniform Sampling như hiện tại) ---
def baseline_extraction(video_path, num_frames=10):
    start_time = time.time()
    cap = cv2.VideoCapture(video_path)
    frames = []
    if not cap.isOpened():
        return frames, 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // num_frames)
    
    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
        if len(frames) == num_frames: break
    cap.release()
    end_time = time.time()
    return frames, end_time - start_time

try:
    # Chạy thử nghiệm Baseline
    baseline_frames, baseline_time = baseline_extraction(video_path, num_frames=52)
    print(f'OpenCV Baseline hoàn thành trong {baseline_time:.2f} giây (Trích xuất {len(baseline_frames)} frames)')
except Exception as e:
    print('Vui lòng cung cấp video_path hợp lệ để test Baseline:', e)

OpenCV Baseline hoàn thành trong 6.01 giây (Trích xuất 52 frames)


In [3]:
# --- P2: SOTA (TransNetV2 / Scene Boundary Detection) ---
try:
    from transnetv2 import TransNetV2
    model = TransNetV2()
    
    start_time = time.time()
    video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(video_path)
    scenes = model.predictions_to_scenes(single_frame_predictions)
    end_time = time.time()
    
    print(f'TransNetV2 SOTA tìm thấy {len(scenes)} phân cảnh trong {end_time - start_time:.2f} giây')
except Exception as e:
    print('Vui lòng upload video thật để test TransNetV2:', e)

[TransNetV2] Using weights from /kaggle/working/TransNetV2/inference/transnetv2-weights/.


I0000 00:00:1785740757.837144      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785740757.840079      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[TransNetV2] Extracting frames from /kaggle/input/datasets/phmthanhhng27/video-l30/video/L30_V003.mp4
[TransNetV2] Processing video frames 8092/8092
TransNetV2 SOTA tìm thấy 64 phân cảnh trong 26.43 giây


In [4]:
# --- Lưu Kết Quả Ra Output Kaggle ---
# Trong Kaggle, mọi file cần lưu lại phải nằm trong thư mục /kaggle/working/
output_dir = '/kaggle/working/extracted_frames'
os.makedirs(output_dir, exist_ok=True)

try:
    # Ví dụ 1: Lưu các frame của Baseline ra file
    for i, frame in enumerate(baseline_frames):
        save_path = os.path.join(output_dir, f'baseline_frame_{i:03d}.jpg')
        cv2.imwrite(save_path, frame)
    print(f'Đã lưu thành công {len(baseline_frames)} frames của Baseline vào {output_dir}')

    # Ví dụ 2: Lưu các frame của SOTA với độ phân giải NGUYÊN GỐC
    sota_output_dir = '/kaggle/working/sota_frames'
    os.makedirs(sota_output_dir, exist_ok=True)
    
    if 'scenes' in locals():
        cap = cv2.VideoCapture(video_path)
        for i, scene in enumerate(scenes):
            start_frame, end_frame = scene
            middle_frame_idx = (start_frame + end_frame) // 2
            
            # Đọc frame gốc từ video bằng OpenCV thay vì dùng mảng downscale của TransNetV2
            cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame_idx)
            ret, frame_bgr = cap.read()
            if ret:
                save_path = os.path.join(sota_output_dir, f'sota_scene_{i:03d}.jpg')
                cv2.imwrite(save_path, frame_bgr)
        cap.release()
        print(f'Đã lưu thành công {len(scenes)} frames nguyên gốc của SOTA vào {sota_output_dir}')

    print('\nBạn có thể xem các ảnh này ở cột Output bên phải Kaggle, hoặc zip thư mục này lại để tải về.')
except Exception as e:
    print('Không có frame nào được lưu (có thể do video path chưa đúng):', e)

Đã lưu thành công 52 frames của Baseline vào /kaggle/working/extracted_frames
Đã lưu thành công 64 frames nguyên gốc của SOTA vào /kaggle/working/sota_frames

Bạn có thể xem các ảnh này ở cột Output bên phải Kaggle, hoặc zip thư mục này lại để tải về.


## Đánh giá:
- So sánh thời gian chạy (Execution Time) giữa 2 đoạn code trên.
- So sánh độ chính xác: SOTA cắt đúng ranh giới cảnh (scene boundary), giúp OCR/VLM phân tích rõ nét hơn thay vì cắt ngẫu nhiên có thể dính frame chuyển cảnh bị nhòe.